<a href="https://colab.research.google.com/github/vzuza/mapa_restec_ceped/blob/main/mapa_residentes_restec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [64]:
import duckdb
import geobr
import geopandas as gpd
import pandas as pd
from unidecode import unidecode
from importlib.metadata import version
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

In [28]:
df = pd.read_excel('/content/Tabela_Municipio_Area_Forca_Residentes.xlsx')

In [29]:
con = duckdb.connect()
con.execute("INSTALL spatial")
con.execute("LOAD spatial")

In [30]:
for raiz, pastas, arquivos in os.walk('/root/.cache'):
    for arquivo in arquivos:
        if 'municipalities_2020_simplified.parquet' in arquivo:
            print(os.path.join(raiz, arquivo))

In [33]:
arquivo = None

for pasta in ['/root/.cache', '/content', '/tmp']:
    for raiz, diretorios, arquivos in os.walk(pasta):
        for nome in arquivos:
            if nome == 'municipalities_2020_simplified.parquet':
                arquivo = os.path.join(raiz, nome)
                break
        if arquivo:
            break
    if arquivo:
        break

print("Arquivo encontrado:", arquivo)

Arquivo encontrado: /tmp/geobr_n4mncug1/municipalities_2020_simplified.parquet


In [34]:
municipios = gpd.read_parquet(arquivo)

print("Total:", len(municipios))
print(municipios.columns.tolist())

Total: 5570
['code_muni', 'name_muni', 'code_state', 'abbrev_state', 'name_state', 'code_region', 'name_region', 'year', 'geometry']


In [35]:
municipios = municipios[
    municipios['abbrev_state'] == 'PR'
].copy()

Municípios do Paraná: 399


In [42]:
municipios['Municipio_padronizado'] = (
    municipios['name_muni']
    .astype(str)
    .str.strip()
    .apply(lambda x: unidecode(x).upper())
)

In [43]:
df['Municipio_padronizado'] = (
    df['MUN']
    .astype(str)
    .str.strip()
    .apply(lambda x: unidecode(x).upper())
)

In [44]:
mapa_dados = municipios.merge(
    df_municipios,
    on='Municipio_padronizado',
    how='left',
    indicator=True
)

In [47]:
print(mapa_dados['_merge'].value_counts())

print(
    mapa_dados['_merge'].value_counts(normalize=True) * 100
)

_merge
left_only     373
both           26
right_only      0
Name: count, dtype: int64
_merge
left_only     93.483709
both           6.516291
right_only     0.000000
Name: proportion, dtype: float64


In [48]:
nao_encontrados = mapa_dados.loc[
    mapa_dados['_merge'] == 'left_only',
    ['name_muni', 'Municipio_padronizado']
]

display(nao_encontrados)

,name_muni,Municipio_padronizado
0,Abatiá,ABATIA
1,Adrianópolis,ADRIANOPOLIS
2,Agudos do Sul,AGUDOS DO SUL
3,Almirante Tamandaré,ALMIRANTE TAMANDARE
4,Altamira do Paraná,ALTAMIRA DO PARANA
...,...,...
394,Alto Paraíso,ALTO PARAISO
395,Doutor Ulysses,DOUTOR ULYSSES
396,Virmond,VIRMOND
397,Vitorino,VITORINO


In [49]:
print(df_municipios['Municipio_padronizado'].sort_values().to_list())

['APUCARANA', 'ARAPONGAS', 'ARAUCARIA', 'CAMPO MOURAO', 'CASCAVEL', 'CIANORTE', 'CORNELIO PROCOPIO', 'CRUZEIRO DO OESTE', 'CURITIBA', 'FOZ DO IGUACU', 'FRANCISCO BELTRAO', 'GUARAPUAVA', 'JACAREZINHO', 'JAGUARIAIVA', 'LONDRINA', 'MARINGA', 'PARANAGUA', 'PARANAVAI', 'PATO BRANCO', 'PIRAQUARA', 'PONTA GROSSA', 'SANTO ANTONIO DA PLATINA', 'SAO JOSE DOS PINHAIS', 'TOLEDO', 'TOTAL', 'UMUARAMA', 'UNIAO DA VITORIA']


In [50]:
def padronizar_municipio(nome):
    nome = unidecode(str(nome)).upper().strip()
    nome = re.sub(r'\s+', ' ', nome)
    return nome

municipios['Municipio_padronizado'] = (
    municipios['name_muni'].apply(padronizar_municipio)
)

df_municipios['Municipio_padronizado'] = (
    df_municipios['Municipio_padronizado'].apply(padronizar_municipio)
)

In [51]:
mapa_dados = municipios.merge(
    df_municipios,
    on='Municipio_padronizado',
    how='left',
    indicator=True
)

In [52]:
df_municipios = df_municipios[
    df_municipios['Municipio_padronizado'] != 'TOTAL'
].copy()

print("Municípios com registros:", len(df_municipios))

Municípios com registros: 26


In [53]:
mapa_dados = municipios.merge(
    df_municipios,
    on='Municipio_padronizado',
    how='left',
    indicator=True
)

In [54]:
mapa_dados['possui_registro'] = (
    mapa_dados['_merge'] == 'both'
)

mapa_dados['residentes'] = (
    mapa_dados['residentes'].fillna(0)
)

In [55]:
print("Total de municípios:", len(mapa_dados))

print(
    "Com registros:",
    mapa_dados['possui_registro'].sum()
)

print(
    "Sem registros:",
    (~mapa_dados['possui_registro']).sum()
)

print(
    "Total de residentes:",
    mapa_dados['residentes'].sum()
)

Total de municípios: 399
Com registros: 26
Sem registros: 373
Total de residentes: 335.0


In [57]:

# Garantir que a geometria esteja correta
mapa_dados = mapa_dados.copy()
mapa_dados = mapa_dados.set_geometry('geometry')

# Identificar municípios com registros
mapa_dados['possui_registro'] = (
    mapa_dados['residentes'].notna()
    | mapa_dados['Forca'].notna()
    | mapa_dados['area_atuacao'].notna()
)

# Valores quantitativos
mapa_dados['residentes'] = pd.to_numeric(
    mapa_dados['residentes'], errors='coerce'
).fillna(0)

# Projetar para SIRGAS 2000 / UTM 22S
mapa_plot = mapa_dados.to_crs(epsg=31982)

print("Municípios:", len(mapa_plot))
print("Com registros:", mapa_plot['possui_registro'].sum())
print("Sem registros:", (~mapa_plot['possui_registro']).sum())

Municípios: 399
Com registros: 399
Sem registros: 0


In [58]:
AZUL_ESCURO = '#17365D'
AZUL_MEDIO = '#3977A8'
AZUL_CLARO = '#A9C7DC'
CINZA = '#E5E9ED'
CINZA_LIMITE = '#FFFFFF'
CINZA_TEXTO = '#526170'
BRANCO = '#FFFFFF'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 9,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'figure.facecolor': BRANCO,
    'axes.facecolor': BRANCO
})

In [59]:
principais = (
    mapa_plot[mapa_plot['possui_registro']]
    .nlargest(10, 'residentes')
)

print(principais[['name_muni', 'residentes']])

             name_muni  residentes
93            Curitiba       123.0
190           Londrina        28.0
208            Maringá        25.0
68            Cascavel        23.0
274       Ponta Grossa        22.0
118      Foz do Iguaçu        20.0
134         Guarapuava        17.0
120  Francisco Beltrão        13.0
386           Umuarama        13.0
89   Cruzeiro do Oeste         8.0
